# 第18周 · 作业：研究实验 5（REINFORCE 的方差缩减）· Capstone 主结果（预计约6小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 实验设计与预注册 | 0.5 小时 |
| B | 实现：滑动平均基线、三种权重、训练函数 | 1.5 小时 |
| C | 执行（3 种设置 × 5 个种子）、结果表、学习曲线 | 1 小时 |
| D | 分析问题 + 实验记录 | 1 小时 |
| E | Capstone 主结果写入实验记录本 | 1.5 小时 |
| F | 代码讲解 | 0.5 小时 |

**约定**：可以使用 AI 编程助手写代码，但一定要写 F 部分的"代码讲解"（用自己的话解释关键代码，并注明哪些部分由 AI 生成），写完再挑一段给自己讲一遍。

**自评**：A–D 用学习指南里的研究实验自评表（设计 30% · 执行 30% · 分析与解释 30% · 表达 10%）；E 看主结果是否 ≥ 3 个种子、是否与基线放在同一张表、能否一键复现。

**运行时间**：C 部分默认每个种子 500 回合（`N_EPISODES = 500`），15 次训练在 CPU 上约 1–2 分钟，不需要 GPU。**写正式结果前把 `N_EPISODES` 改成 1000 再跑一遍**（Colab CPU 约 3–5 分钟），并在实验记录里写明你用的回合预算。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：Gymnasium（强化学习环境库；只训练、不渲染画面，所以不需要 pygame）
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium>=1.4,<2"], check=True)
print("Gymnasium 已就绪")

In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import gymnasium as gym
from zero2researcher import experiments
from zero2researcher.checks import week18 as ck
from zero2researcher.checks import report

torch.set_num_threads(1)     # RL 全部在 CPU 上跑：小网络 + 单个状态时单线程最快，结果也与线程数无关
print("torch", torch.__version__, "| gymnasium", gym.__version__, "| device: cpu")

## A · 实验设计（先写，再跑）

**研究问题**：在 CartPole-v1 上，给 REINFORCE 减去一个基线、再把优势标准化，能不能让它更快、更可靠地达标？

讲义 S3 里你已经看到：无基线的 REINFORCE 在 5 个种子之间像彩票。本实验比较三种"乘在 $\log\pi_\theta(a_t \mid s_t)$ 前面的权重"——**只改这一处，其余全部相同**：

| 设置 | 每一步的权重 $w_t$ | 说明 |
| --- | --- | --- |
| **A** 无基线 | $G_t$ | 讲义 S3 原样 |
| **B** 滑动平均基线 | $G_t - b_t$ | $b_t$ = **过去各回合**在第 $t$ 步的 reward-to-go 的指数滑动平均（只依赖过去，因此无偏） |
| **C** B + 优势标准化 | $\dfrac{\mathrm{adv}_t - \mathrm{mean}(\mathrm{adv})}{\mathrm{std}(\mathrm{adv}) + \epsilon}$，$\mathrm{adv}_t = G_t - b_t$ | 再把这一回合的优势（advantage）减均值、除以标准差 |

$b_t$ 的更新（每个回合结束、**先用旧的 $b$ 算完权重之后**）：对这一回合走过的每个时间步 $t < T$，

$$b_t \leftarrow b_t + \beta\,(G_t - b_t), \qquad \beta = 0.05$$

又是"旧值 + 步长 × (目标 − 旧值)"。它记住的是"策略最近在第 $t$ 步通常还能再拿多少分"，于是 $G_t - b_t > 0$ 的意思是"这一步之后比最近的平常表现更好"。

**固定**（与讲义 S3 相同，来自 HF Unit 4）：策略网络 4 → 16 → 2、Adam lr 1e-2、$\gamma = 1.0$、每玩一局更新一次、reward-to-go；β = 0.05；每个种子 `N_EPISODES` 个回合；种子 0–4（RL 的种子间方差很大，本实验用 5 个种子）。

**指标**：

- **首次达标回合数**：最近 100 回合平均回报首次 ≥ 475 时的回合编号（475 是 Gymnasium 注册表里 CartPole-v1 的 `reward_threshold`）；预算内没达标记为 "> 预算"；
- **最终表现**：最后 100 回合的平均回报（均值 ± 标准差，5 个种子）——用来发现"达标之后又崩掉"；
- **学习曲线**：5 个种子的均值 ± 标准差带。

**提前告诉你**：凭直觉，很多人会预测 A < B < C（C 最好）。**C 不一定比 B 好**。别因此改写你的预期——把你真实的预期写下来，然后让数据说话。

**A0 · 预注册**（跑任何代码之前写下，跑完不许改）

- 我预期三个设置按"首次达标回合数"的排序是：
- 理由（各一句）：
- 什么样的结果会推翻我的预期：

## B · 实现

下面这些直接沿用讲义 S3（读懂即可）：策略网络、玩一局、reward-to-go、首次达标回合数、滑动平均曲线。

In [ ]:
GAMMA, LR, BETA = 1.0, 1e-2, 0.05
MAX_T = 500                                    # CartPole-v1 每回合最多 500 步：b 需要覆盖所有可能的时间步
N_EPISODES = 500                               # 快速版；正式结果改成 1000
EPS = np.finfo(np.float32).eps.item()          # HF Unit 4 原文的写法：标准化时防止除以 0


class Policy(nn.Module):
    # HF Unit 4 的 CartPole 策略网络：4 → 16 → 2，输出向左 / 向右的概率（讲义 S3 原样）
    def __init__(self, s_size=4, h_size=16, a_size=2):
        super().__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        return torch.softmax(self.fc2(torch.relu(self.fc1(x))), dim=-1)


def run_episode(env, policy):
    # 用当前策略玩一局，返回这一局的状态 (T, 4)、动作 (T,)、奖励 (T,)（讲义 S3 原样）
    obs, info = env.reset()
    states, actions, rewards = [], [], []
    done = False
    while not done:
        with torch.no_grad():
            probs = policy(torch.as_tensor(obs, dtype=torch.float32))
        action = int(torch.multinomial(probs, 1))
        states.append(obs)
        actions.append(action)
        obs, reward, terminated, truncated, info = env.step(action)
        rewards.append(reward)
        done = terminated or truncated
    return np.array(states, dtype=np.float32), np.array(actions), np.array(rewards)


def reward_to_go(rewards, gamma):
    # G_t = r_t + γ r_{t+1} + …（讲义 S3 / 练习 E4）
    G = np.zeros(len(rewards))
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        G[t] = running
    return G


def episodes_to_solve(returns, threshold=475, window=100):
    # 首次"最近 window 回合平均回报 ≥ threshold"时的回合编号（从 1 数）；没达到返回 None（讲义 S3 原样）
    for i in range(window, len(returns) + 1):
        if np.mean(returns[i - window:i]) >= threshold:
            return i
    return None


def moving_average(x, k=20):
    return np.convolve(x, np.ones(k) / k, mode="valid")

**B1** · 写 `update_baseline(b, G, beta)`：`b` 是长度 `MAX_T` 的数组，`G` 是这一回合的 reward-to-go（长度 $T$）。只更新前 $T$ 个时间步：$b_t \leftarrow b_t + \beta (G_t - b_t)$，并 `return b`。

In [ ]:
def update_baseline(b, G, beta=BETA):
    # 在这里写你的代码
    raise NotImplementedError


b_demo = update_baseline(np.zeros(6), np.array([4.0, 3.0, 2.0, 1.0]), beta=0.5)
print("一个 4 步回合之后（β = 0.5）:", b_demo)
ck.moving_baseline(update_baseline)

**B2** · 写 `compute_weights(G, b, setting)`：按上面的表格返回这一回合每一步的权重 $w_t$（numpy 数组，长度 $T$）。`setting` 是 `"A"`、`"B"` 或 `"C"`；`b` 是**还没用这一回合更新过**的基线，只用它的前 $T$ 个元素。C 的标准化用 `EPS`：`(adv - adv.mean()) / (adv.std() + EPS)`。

In [ ]:
def compute_weights(G, b, setting):
    T = len(G)
    # 在这里写你的代码
    raise NotImplementedError


G_demo, b_demo = np.array([4.0, 3.0, 2.0, 1.0]), np.array([2.0, 2.0, 1.0, 1.0, 9.0, 9.0])
w = {s: compute_weights(G_demo, b_demo, s) for s in "ABC"}
for s, ws in w.items():
    print(f"{s}: {np.round(ws, 3)}")
report("三种权重 A / B / C",
       np.allclose(w["A"], G_demo) and np.allclose(w["B"], [2, 1, 1, 0])
       and abs(w["C"].mean()) < 1e-6 and abs(w["C"].std() - 1) < 1e-3,
       "A 应等于 G；B 应等于 G − b[:T] = [2, 1, 1, 0]；C 应是 B 标准化之后的结果（均值 0、标准差 1）",
       "B 只用 b 的前 T 个元素：b[:len(G)]；C：(adv − adv.mean()) / (adv.std() + EPS)")

**B3** · 写 `train_reinforce(setting, seed, n_episodes)`，返回每个回合的回报（列表，长度 `n_episodes`）。每个回合：

1. `run_episode` 玩一局，`reward_to_go` 算 $G$；
2. 用**旧的** `b` 算权重 `w = compute_weights(G, b, setting)`；
3. **再**更新 `b = update_baseline(b, G)`（A 不用 `b`，更新了也无妨）；
4. 损失 $-\sum_t \log\pi_\theta(a_t \mid s_t)\, w_t$，三步更新（与讲义 S3 的训练循环相同，只是把 `G` 换成 `w`）；
5. 记下这一局的回报 `rewards.sum()`。

关于第 2、3 步的顺序：先用旧的 `b` 算、再更新，是保证"基线只依赖过去回合"的好习惯。不过在这个滑动平均写法下，把两行对调其实**不会引入偏差**——F 部分会请你推一推为什么（提示：$b'_t = b_t + \beta(G_t - b_t)$，用 $G_t - b_t$ 表示 $G_t - b'_t$），不信也可以对调两行跑一下。真正会引入偏差的，是让基线依赖**当前这条轨迹**的写法，例如 HF Unit 4 的"回合内平均回报" $\mathrm{mean}_t(G_t)$：它用整局的 $G$ 算出，里面包含了 $a_t$ 之后发生的一切（D 部分问题 3）。

In [ ]:
def train_reinforce(setting, seed, n_episodes=N_EPISODES, gamma=GAMMA, lr=LR, beta=BETA):
    experiments.set_seed(seed)                 # 先固定种子再建网络
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)                       # 只在开头给环境播一次种
    policy = Policy()
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    b = np.zeros(MAX_T)
    returns = []
    # 在这里写你的代码
    raise NotImplementedError
    return returns


smoke = train_reinforce("B", seed=0, n_episodes=30)
print("冒烟测试（B，种子 0，30 回合）最后 5 局的回报:", smoke[-5:])

## C · 执行与结果

**C1** · 主循环：3 种设置 × 5 个种子，结果存进 `results`。要求的结构（自查会检查）：

```python
results["A"] = [种子 0 的回报列表, 种子 1 的, …, 种子 4 的]     # 每个列表长度 N_EPISODES
results["B"] = [...]
results["C"] = [...]
```

跑完之后结果会存成 JSON 文件（`outputs/week18_reinforce_results.json`），写实验记录、画图时不必重跑。

In [ ]:
SETTINGS, SEEDS = ("A", "B", "C"), (0, 1, 2, 3, 4)
LABELS = {"A": "A: no baseline", "B": "B: moving-average baseline", "C": "C: B + advantage normalisation"}
results = {}
t0 = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"{len(SETTINGS)} 种设置 × {len(SEEDS)} 个种子 × {N_EPISODES} 回合，共用时 {time.time() - t0:.0f} 秒")

os.makedirs("outputs", exist_ok=True)
with open(os.path.join("outputs", "week18_reinforce_results.json"), "w") as f:
    json.dump({"n_episodes": N_EPISODES, "gamma": GAMMA, "lr": LR, "beta": BETA, "seeds": list(SEEDS),
               "returns": results}, f)
ck.reinforce_results(results)

**C2** · 结果表 `table`（pandas DataFrame）：每个设置一行；列为"预算内达标的种子数"（如 `5/5`）、"各种子的首次达标回合"（没达标写 `>500` 之类）、"达标回合中位数"（没达标的种子按"无穷大"参与排序，中位数是无穷大时写 `>预算`）、"最后 100 回合平均回报（均值 ± 标准差）"（用 `experiments.format_result`）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
table

**C3** · 两张图，标题写结论：

1. 学习曲线：横轴回合，纵轴回报（20 回合滑动平均）；每个设置画 5 个种子的**均值**线和**均值 ± 标准差**的阴影带；画出 475 的虚线。
2. 分种子的曲线：3 个子图（每个设置一个），每个子图里 5 条线（每个种子一条）。均值带会把"达标之后又崩掉"抹平，这张图能看出来。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


## D · 分析与实验记录

**D1 · 分析问题**（每题三五句话，引用你表里的数字）

1. 按"首次达标回合数"给三个设置排序。B 与 A 的差距大于种子间的波动吗？你的预注册被支持还是被推翻了？
2. **C 为什么可能不比 B 好？** 提示：(a) Adam 本身已经按梯度的历史大小自动调整每个参数的步长，再把权重的尺度归一化，还能带来多少好处？(b) 标准化又减了一次**这一回合**优势的均值——这个均值依赖当前这条轨迹，它对"好回合"和"坏回合"分别做了什么（想一想：一局 50 步的回合和一局 500 步的回合，标准化之后权重的平均都是 0）？
3. **HF 写法的偏差。** HF Unit 4 的原文是 `(G - G.mean()) / (G.std() + eps)`，相当于用"回合内的平均回报"当基线。在 $\gamma = 1$ 的 CartPole 里 $G_t = T - t$，写出 $G_t - \mathrm{mean}_t(G_t)$。它还和"这一局坚持得久不久"有关吗？按 Spinning Up Part 3 的说法，基线只能依赖什么，梯度的期望才不变？回合内均值为什么不满足？
4. 只报告"首次达标回合数"会掩盖什么？用第二张图里的具体种子说明。
5. 5 个种子够不够让你说"B 比 A 好"？如果要把这个结论写进论文，你还会做什么（更多种子、报告四分位距、bootstrap 置信区间……）？

**选做**：把 HF 的写法当作第四个设置 D（权重 `(G - G.mean()) / (G.std() + EPS)`），在 $\gamma = 1.0$ 与 $\gamma = 0.99$ 下各跑 5 个种子，与 A、B 比较；你的结果支持问题 3 的分析吗？

（在这里回答）

**D2 · 实验记录**（正式格式，"结果"一栏写均值 ± 标准差并引用上面的表和图；"局限"一栏讨论固定超参数——学习率、γ、β、回合预算——对结论的限定）

## 实验记录 #__ · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## E · Capstone 主结果（写入实验记录本）

本周的 Capstone 里程碑是**主结果**：用第 12 周设计文档里定下的主实验配置，跑出**至少 3 个种子**（RL 类选题建议 5 个）的结果，与第 16 周的基线放进同一张表。要求：

1. 主结果表：每个方法一行，主指标写**均值 ± 标准差（n = 种子数）**；基线与你的方法在**同一划分、同一预算**下比较；
2. 主图：带误差棒（或均值 ± 标准差带）的图，标题写结论；
3. 可复现：注明生成这些数字的 notebook 与运行配置；固定种子；把结果存成文件（像 C1 那样存 JSON 或 CSV），下周画消融图时直接读取；
4. 与设计文档的偏差：写明这次的配置与第 12 周设计相比改了什么、为什么。

按下面的模板在你的实验记录本里新开一条（字段与 `templates/lab_notebook.md` 相同，外加主结果专用的几项），并在这里粘一份。

## 实验记录 #__ · Capstone 主结果 · 日期：____

- 研究问题（与第 12 周设计文档一致；如有修改，写明改了什么、为什么）：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线（第 16 周的基线结果，原样引用）：
- 数据与划分方式（如何避免泄漏）：
- 随机种子（≥ 3 个；RL 类选题建议 5 个）：
- 结果（数值：均值±标准差；图：链接或插图）：

| 方法 | 主指标（均值 ± 标准差，n = __） | 次要指标 | 单次运行时间 |
| --- | --- | --- | --- |
| 基线 |  |  |  |
| 我的方法 |  |  |  |

- 复现方式（notebook 文件名或链接、运行配置、结果文件名、一键运行需要多久）：
- 与设计文档的偏差：
- 解释：
- 局限：
- 下一步（第 19 周的消融：拿掉哪个部件、预期会怎样）：

## F · 代码讲解（每份作业都写）

用你自己的话写一段（约 150–300 字）：

1. 逐段解释 `train_reinforce`——尤其是：如果把 `compute_weights` 与 `update_baseline` 两行对调，B 的权重会变成什么（用 $G_t - b_t$ 表示 $G_t - b'_t$）？这会不会引入偏差？HF 的回合内均值为什么就会？为什么 `b` 要按时间步分别记录，而不是记一个数？
2. `probs[torch.arange(len(actions)), torch.as_tensor(actions)]` 取出的是什么？为什么损失前面有负号？
3. 注明哪些代码由 AI 助手生成、你做了哪些检查或修改（例如：种子是否在建网络之前设置、`b` 是否只更新前 T 个元素、记录的是不是每局的奖励之和）。

讲不清楚的代码视同不是你写的——写完后挑一段给自己讲一遍，讲不顺的地方就是要回头弄懂的地方。

（在这里写）